## 0. Local Infrastructure Setup

To ensure absolute compliance with enterprise data privacy standards and eliminate execution costs, this entire curriculum operates locally using **Ollama** running an optimized `llama3` instance. Run the following cells to initialize your local environment.

In [1]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,066 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy In

In [2]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Step 3: Spin up the Ollama background host process
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)  # Verify backend execution completes system setup
print("Ollama engine successfully mounted on: http://localhost:11434")

Ollama engine successfully mounted on: http://localhost:11434


In [4]:
# Step 4: Pull down the official local 'llama3' execution engine weights
!ollama pull llama3

###  Vector Search & Retrieval Setup

**Vector Database**, generate **Embeddings**, and assemble the final **RAG Architecture**. We will use `chromadb` (a lightweight vector database) and Ollama's `nomic-embed-text` for creating embeddings directly within your local setup.

In [5]:
# Step 5: Establish runtime interfaces & parameterized execution wrapper
import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, system_prompt=None, model="llama3", temperature=0.7, stream=False):
    """
    Encapsulates raw HTTP infrastructure interactions into an atomic, variable-controlled interface.
    Allows strict separation of system intent instructions from raw operational runtime context inputs.
    """
    # Handle incoming system configuration contexts using standard parameter patterns
    combined_prompt = f"System Context:\n{system_prompt}\n\nUser Task:\n{prompt}" if system_prompt else prompt

    payload = {
        "model": model,
        "prompt": combined_prompt,
        "stream": stream,
        "options": {
            "temperature": temperature
        }
    }

    try:
        resp = requests.post(OLLAMA_API_URL, json=payload)
        if resp.status_code != 200:
            raise RuntimeError(f"Execution gateway error: {resp.status_code} - {resp.text}")

        if stream:
            for line in resp.text.splitlines():
                if not line.strip(): continue
                data_chunk = json.loads(line)
                print(data_chunk.get("response", ""), end="")
            print("\n")
            return None
        else:
            return resp.json().get("response", "")
    except Exception as e:
        return f"System Connection Failure Encountered: {str(e)}"

print("Execution environment fully functional. System status: READY.")

Execution environment fully functional. System status: READY.


In [6]:
# Step 6: Install Vector Database dependencies
# ChromaDB is an open-source vector database perfect for local RAG implementations.
!pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [7]:
# Step 7: Pull a specialized Embedding Model
# We use 'nomic-embed-text' to convert our documents into numerical vectors.
!ollama pull nomic-embed-text

In [8]:
# Step 8: Create the "Company Policy" Dataset
# For this case study, we will use a small list of text chunks representing an employee handbook.
company_policies = [
    "Policy 101: Remote Work. Employees can work remotely up to 3 days a week. Mandatory core hours are 10 AM to 3 PM.",
    "Policy 102: Travel Expenses. Flights under 4 hours must be economy. Meals are capped at $75 per day.",
    "Policy 103: Paid Time Off (PTO). Employees accrue 1.5 days of PTO per month. Max carryover to the next year is 5 days.",
    "Policy 104: IT Security. Passwords must be changed every 90 days. USB drives are strictly prohibited on company devices.",
    "Policy 105: Code of Conduct. Harassment of any kind is subject to immediate termination. All complaints should go to HR directly."
]
print("Company policy dataset loaded.")

Company policy dataset loaded.


In [9]:
# Step 9: Initialize ChromaDB and Embed the Documents
import chromadb

# 1. Initialize the vector database (in-memory for this tutorial)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="company_handbook")

In [10]:
# 2. Define a function to hit Ollama's embedding endpoint
def get_embedding(text, model="nomic-embed-text"):
    url = "http://localhost:11434/api/embeddings"
    payload = {"model": model, "prompt": text}
    response = requests.post(url, json=payload)
    return response.json().get("embedding", [])

In [11]:
# 3. Generate embeddings and store them in ChromaDB
print("Generating embeddings and loading vector database...")
for i, doc in enumerate(company_policies):
    vector = get_embedding(doc)
    collection.add(
        ids=[f"doc_{i}"],
        embeddings=[vector],
        documents=[doc],
        metadatas=[{"source": f"Policy_Page_{i+1}"}]
    )
print("Database successfully populated with vector embeddings!")

Generating embeddings and loading vector database...
Database successfully populated with vector embeddings!


In [12]:
vector

[1.6363445520401,
 0.004321828950196505,
 -3.4883534908294678,
 -0.7954112887382507,
 1.0703153610229492,
 0.7272853255271912,
 0.7452820539474487,
 0.25098440051078796,
 0.30976682901382446,
 0.776272714138031,
 -0.4482914209365845,
 0.9395400285720825,
 0.3211466372013092,
 0.4565255343914032,
 0.484057754278183,
 -1.0592191219329834,
 0.9472771883010864,
 -0.4975331127643585,
 -1.6656575202941895,
 0.22967936098575592,
 -0.058567114174366,
 -0.7439938187599182,
 -0.28381457924842834,
 0.011739557608962059,
 1.3449290990829468,
 1.2592501640319824,
 1.2317230701446533,
 0.4706282317638397,
 0.7122861742973328,
 -0.27597159147262573,
 -0.7284867167472839,
 1.2241157293319702,
 0.33763182163238525,
 -0.612640917301178,
 -0.4374276399612427,
 -1.215983510017395,
 0.744154691696167,
 -1.0601330995559692,
 -0.7509269118309021,
 -0.2259417623281479,
 1.3818752765655518,
 0.18288078904151917,
 0.5983514785766602,
 -0.9363844394683838,
 0.42615026235580444,
 -0.10868121683597565,
 0.63038271

### The RAG Architecture

We will build a pipeline that takes a user query, converts it to an embedding, searches the Vector DB, and passes the retrieved context to `llama3` using the `call_ollama` wrapper you built in Step 5.

In [13]:
# Step 10: The Vector Search Phase (Retrieval)
# Let's test the retrieval engine before attaching the LLM.
test_query = "What are the rules for flying and eating on business trips?"
query_vector = get_embedding(test_query)

In [14]:
# Retrieve the top 2 most mathematically similar chunks
search_results = collection.query(
    query_embeddings=[query_vector],
    n_results=2
)

In [15]:
print(f"User Query: '{test_query}'\n")
print("Top Retrieved Contexts:")
for idx, retrieved_doc in enumerate(search_results['documents'][0]):
    print(f"{idx + 1}. {retrieved_doc}")

User Query: 'What are the rules for flying and eating on business trips?'

Top Retrieved Contexts:
1. Policy 102: Travel Expenses. Flights under 4 hours must be economy. Meals are capped at $75 per day.
2. Policy 103: Paid Time Off (PTO). Employees accrue 1.5 days of PTO per month. Max carryover to the next year is 5 days.


In [16]:
# Step 11: The Complete RAG Pipeline (Retrieve -> Augment -> Generate)
def ask_hr_bot(user_question):
    """
    Implements the full RAG lifecycle.
    """
    # Phase 1: Retrieve
    query_vector = get_embedding(user_question)
    search_results = collection.query(query_embeddings=[query_vector], n_results=2)
    retrieved_contexts = search_results['documents'][0]

    # Combine retrieved chunks into a single string
    context_string = "\n".join(retrieved_contexts)

    # Phase 2: Augment (Inject context into the system prompt)
    system_prompt = f"""You are a strict but helpful corporate HR assistant.
    Answer the user's question using ONLY the provided policy context below.
    If the answer is not contained in the context, explicitly state: "I cannot find the answer in the company policies." Do not invent rules.

    CONTEXT:
    {context_string}
    """

    # Phase 3: Generate
    print(f"User Asked: {user_question}")
    print("Searching vector database and generating response...\n")

    # Notice we lower the temperature to 0.1 for more factual, grounded responses
    answer = call_ollama(
        prompt=user_question,
        system_prompt=system_prompt,
        model="llama3",
        temperature=0.1
    )

    print(f"HR Bot:\n{answer}")

In [17]:
# Step 12: Run the Case Study!
# 1. A question directly answered in the text
ask_hr_bot("Can I use a thumb drive to move my presentations?")

User Asked: Can I use a thumb drive to move my presentations?
Searching vector database and generating response...

HR Bot:
I'm happy to help! According to Policy 104: IT Security, USB drives are strictly prohibited on company devices. Therefore, you cannot use a thumb drive to move your presentations.


In [18]:
# 2. A question requiring light synthesis
ask_hr_bot("If I work at the company for 3 months, how much PTO will I have accrued?")

User Asked: If I work at the company for 3 months, how much PTO will I have accrued?
Searching vector database and generating response...

HR Bot:
According to Policy 103: Paid Time Off (PTO), employees accrue 1.5 days of PTO per month.

Since you've worked at the company for 3 months, let's calculate your accrued PTO:

3 months × 1.5 days/month = 4.5 days

So, after 3 months, you will have accrued 4.5 days of PTO.


In [19]:
# 3. A question meant to trigger the guardrails (not in the text)
ask_hr_bot("Are we allowed to bring dogs into the office?")

User Asked: Are we allowed to bring dogs into the office?
Searching vector database and generating response...

HR Bot:
I cannot find the answer in the company policies.
